# 6교시 · 도전: T4로 8B QLoRA · Day 1 정리

> **VLM 경량화 2일 과정 · Day 1 (6교시) · 도전/정리**
> 실습 환경: **Google Colab (T4 GPU)** · 모델: **Qwen/Qwen3-VL-8B-Instruct** · 데이터: **HuggingFaceM4/ChartQA**

---

## 이 교시의 목표
- **8B를 4bit로 T4에 올려** 추론·학습이 되는지 한계를 시험한다(batch1·해상도↓·grad accum).
- **OOM**이 났을 때의 대처를 체득한다.
- **4B vs 8B**(메모리·속도·품질) 트레이드오프를 표로 정리한다.
- Day 2(AWQ·서빙) 흐름을 예고하고 **이관 체크리스트**를 점검한다.

> ⚠️ **도전 과제**입니다. 환경에 따라 OOM이 날 수 있고, 그 경험 자체가 학습 목표입니다. 4교시(4B QLoRA)는 안전한 기본값, 이번은 "어디까지 되나"의 탐색입니다.


## 0. 공통 헤더 — Google Drive(VLM_FT_2) 마운트 + HF_TOKEN 로드

> 📌 **모든 Day 1 노트북은 이 셀을 먼저 실행합니다.** Google Drive의 작업 폴더 **`VLM_FT_2`** 를 마운트하고, `.env`의 **HF_TOKEN**을 불러옵니다.
> - `VLM_DIR` / `DATA_DIR` : 교시 간 공유 폴더(전처리 데이터·어댑터·결과가 여기 모입니다).
> - **HF_TOKEN**: `VLM_FT_2/.env` 에 `HF_TOKEN=hf_...` 한 줄을 넣어두면 자동 로드됩니다(다운로드 경고 방지·비공개 모델 접근). `login()`은 호출하지 않습니다(환경변수만으로 충분, 경고 방지).

In [1]:
# ════════════════════════════════════════════════════════════
#  공통 헤더 · Google Drive(VLM_FT_2) 마운트 + HF_TOKEN(.env) 로드
#  (모든 Day1 노트북 상단에서 동일하게 실행)
# ════════════════════════════════════════════════════════════
import os

# (1) Google Drive 마운트 → 작업 폴더 VLM_FT_2 (교시 간 데이터·어댑터·결과 공유)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    VLM_DIR = '/content/drive/MyDrive/VLM_FT_2'      # Drive 경로(권장)
except Exception:
    VLM_DIR = '/content/VLM_FT_2'                     # Colab 아니면 로컬 폴백
DATA_DIR = f'{VLM_DIR}/data'
os.makedirs(DATA_DIR, exist_ok=True)

# (2) .env 에서 HF_TOKEN 로드. login()은 부르지 않음(환경변수만으로 인증, 경고 없음).
try:
    from dotenv import load_dotenv
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'python-dotenv'])
    from dotenv import load_dotenv
ENV_PATH = f'{VLM_DIR}/.env'
if os.path.exists(ENV_PATH):
    load_dotenv(ENV_PATH)
    print('HF_TOKEN:', '로드됨' if os.environ.get('HF_TOKEN') else '없음')
else:
    print(f'.env 없음 — {ENV_PATH} 에 HF_TOKEN=hf_... 한 줄을 만들면 자동 로드됩니다(공개 데이터만 쓰면 없어도 됨)')
print('작업 폴더 VLM_DIR =', VLM_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
HF_TOKEN: 로드됨
작업 폴더 VLM_DIR = /content/drive/MyDrive/VLM_FT_2


## 1. 8B를 4bit로 — 가능한가? 먼저 계산

1교시 계산을 8B에 다시 적용합니다.

| 항목 | 값 |
|---|---|
| 8B fp16 가중치 | ≈ 16.3 GB → **T4 초과**(못 올림) |
| 8B int4 가중치 | ≈ 4.1 GB → **적재 가능** |

가중치만 보면 4bit 8B는 T4에 들어갑니다. 문제는 **학습 시 추가로 드는** 활성값·그래디언트·옵티마이저 상태입니다. 그래서 **batch=1, 해상도↓, grad accum, grad checkpointing**을 총동원합니다.

In [2]:
!pip install -q -U "transformers>=4.57" "accelerate>=1.0" "bitsandbytes>=0.44" "datasets>=3.0" "peft>=0.13" "qwen-vl-utils>=0.0.8" "python-dotenv" "pillow<12"
import torch, transformers
print('transformers:', transformers.__version__)

transformers: 5.12.1


## 2. 8B 4bit 로드 + VRAM 측정

4B보다 해상도 상한을 더 낮춰(`128*28*28`) 이미지 토큰을 줄입니다.

In [3]:
# ── 8B 모델을 4bit로 로드하는 셀 ─────────────────────────────────────────────
# 이미 transformers/torch는 위 셀에서 설치/임포트되어 있으므로 여기서는 필요한 클래스만 가져옵니다.
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

# 사용할 베이스 모델 ID (Qwen3-VL 8B Instruct)
MODEL_ID_8B = 'Qwen/Qwen3-VL-8B-Instruct'   # 약 8,767.1M 파라미터

# 8B는 메모리 여유가 적으므로, 이미지 토큰 수를 줄이기 위해 해상도 상한을 더 낮게 설정
# (Qwen 계열 비전 입력은 28x28 단위 패치 기준으로 픽셀 범위를 제어)
MIN_PIXELS, MAX_PIXELS = 64 * 28 * 28, 128 * 28 * 28

# Processor 로드:
# - min_pixels/max_pixels로 비전 입력 크기 범위를 제한해 VRAM 사용량을 안정화
processor = AutoProcessor.from_pretrained(
    MODEL_ID_8B,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS
)

# 학습/패딩 일관성을 위해 우측 패딩 사용
processor.tokenizer.padding_side = 'right'

# bitsandbytes 4bit 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                  # 4bit 가중치 로드
    bnb_4bit_quant_type='nf4',          # QLoRA에서 흔히 쓰는 NF4 양자화 타입
    bnb_4bit_use_double_quant=True,     # 더 효율적인 이중 양자화
    bnb_4bit_compute_dtype=torch.float16,  # 연산 dtype은 fp16
)

# 피크 메모리 통계를 리셋하고, 모델 로드 후 현재 VRAM 점유량 측정
torch.cuda.reset_peak_memory_stats()

# 모델 로드:
# - quantization_config: 4bit 로드 적용
# - device_map='auto': 가능한 장치(GPU)에 자동 배치
# - dtype=torch.float16: 비양자화 텐서/연산 기본 dtype
# - attn_implementation='sdpa': PyTorch SDPA 어텐션 사용
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID_8B,
    quantization_config=bnb_config,
    device_map='auto',
    dtype=torch.float16,
    attn_implementation='sdpa',
)

# 현재 할당된 VRAM(GB) 출력
load_vram = torch.cuda.memory_allocated() / 1024**3
print(f'8B 4bit 로드 후 VRAM: {load_vram:.2f} GB')

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

8B 4bit 로드 후 VRAM: 5.96 GB


### 🔍 결과 해석 — 8B 4bit 로드
- `8B 4bit 로드 후 VRAM: 5.96 GB` → 8.77B 모델을 4bit로 올리니 **약 6GB**입니다. 가중치 4.1GB + 비전 타워·임베딩·lm_head(fp16) + 버퍼를 더한 값으로, **T4(16GB)에 추론용으로는 충분히** 들어갑니다.
- `bitsandbytes ... FutureWarning`은 정상입니다. 학습은 여기에 그래디언트·옵티마이저·활성값이 더 붙어 훨씬 빡빡해집니다(아래에서 확인).

In [4]:
# ── 로드만으로 추론이 되는지 빠른 확인 ─────────────────────────────
# 목적:
# - 방금 4bit로 올린 8B 모델이 "학습 전에도" 추론은 되는지 확인
# - 이후 학습 셀에서 재사용할 헬퍼(SYSTEM, shrink, to_messages, process_vision_info)도 여기서 준비

from datasets import load_dataset, load_from_disk

# 검증용 소규모 샘플 20개 준비
# - 전처리해 저장한 val 셋이 있으면 그것을 우선 사용
# - 없으면 Hugging Face의 ChartQA val split에서 바로 20개 샘플링
_vp = f'{DATA_DIR}/chartqa_val_300'
val_set = (
    load_from_disk(_vp).select(range(20))
    if os.path.isdir(_vp)
    else load_dataset('HuggingFaceM4/ChartQA', split='val').shuffle(seed=42).select(range(20))
)

# ── 시스템 프롬프트 + 메시지/이미지 헬퍼 ────────────────────────────
from PIL import Image
from qwen_vl_utils import process_vision_info  # 메시지 안의 image 항목을 모델 입력 형태로 정리

# 답변을 장황하게 하지 않고, 숫자/짧은 단어로만 출력하도록 유도
SYSTEM = "당신은 차트를 읽고 질문에 답하는 도우미입니다. 설명 없이 한 단어 또는 숫자로만 짧게 답하세요."

def shrink(img, max_side=768):
    """이미지 긴 변을 max_side 이하로 줄이고 RGB로 통일."""
    w, h = img.size
    s = max_side / max(w, h)                    # 긴 변 기준 축소 비율
    if s < 1:                                   # 원본이 큰 경우에만 축소
        img = img.resize((int(w * s), int(h * s)), Image.LANCZOS)
    return img.convert("RGB")                   # 어떤 모드든 RGB로 맞춤

def to_messages(image, question, answer=None):
    """한 샘플을 Qwen 채팅 형식 메시지 리스트로 변환."""
    msgs = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
        {"role": "user", "content": [
            {"type": "image", "image": image},  # 차트 이미지
            {"type": "text", "text": question}  # 사용자 질문
        ]},
    ]
    if answer is not None:
        # answer가 있으면 학습용 메시지(정답 포함)
        msgs.append({"role": "assistant", "content": [{"type": "text", "text": answer}]})
    return msgs                                  # answer가 없으면 추론용 프롬프트

# 학습과 같은 프롬프트 형식으로 추론해야 train/eval 조건이 일치함
@torch.no_grad()                                 # 추론 시 그래디언트 계산 비활성화 → 메모리 절약
def predict(model, image, question, max_new_tokens=32):
    # 1) system + user(image, question) 메시지 구성
    msgs = to_messages(shrink(image), question, None)

    # 2) Qwen 채팅 템플릿 문자열로 변환
    #    add_generation_prompt=True → assistant에 이어서 답하게 만드는 프롬프트
    text = processor.apply_chat_template(
        msgs,
        tokenize=False,
        add_generation_prompt=True
    )

    # 3) 메시지 안의 이미지 정보를 processor가 받을 형식으로 정리
    img_inputs, _ = process_vision_info(msgs)

    # 4) 텍스트/이미지를 모델 입력 텐서로 변환 후, 모델이 올라간 장치(GPU)로 이동
    inputs = processor(
        text=[text],
        images=img_inputs,
        return_tensors="pt"
    ).to(model.device)

    # 5) 생성 실행
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False                          # 샘플링 없이 결정적 출력
    )

    # 6) 입력 프롬프트 이후에 새로 생성된 토큰만 잘라서 디코딩
    gen = out[0][inputs["input_ids"].shape[1]:]
    return processor.decode(gen, skip_special_tokens=True).strip()

# 첫 번째 검증 샘플로 즉시 확인
ex = val_set[0]
print('Q:', ex['query'])
print('정답:', ex['label'][0], '| 8B 예측:', predict(model, ex['image'], ex['query']))

Q: What was the value of the pizza delivery market in the UK at the end of 2017?


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


정답: 6.2 | 8B 예측: 2.1


### 🔍 결과 해석 — 로드만으로 추론 확인
- 8B가 4bit로 올라간 상태에서 **바로 추론이 됩니다**(학습 전). `정답: 6.2 | 8B 예측: 2.1` — 4B와 마찬가지로 **수치 읽기**는 아직 틀립니다(파인튜닝 전이라 당연).
- 핵심은 *"8B도 T4에서 추론 자체는 된다"* 를 확인한 것입니다. 진짜 관문은 **학습**인데, 그건 다음 셀에서.

## 3. 8B QLoRA 학습 — 최소 설정으로 도전

실제 학습이 **시작되어 loss가 흐르는지**가 관건입니다. 수업용으로 **짧게(max_steps=30)** 돌려 메모리만 확인합니다.

극한 설정: `batch=1` · `grad_accum=8` · `해상도 128패치` · `grad checkpointing` · `paged_adamw_8bit`.

In [5]:
# ── 학습 준비: 데이터 로드 + PEFT/LoRA 설정 ──
from datasets import load_dataset, load_from_disk
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

# 전처리된 ChartQA train 데이터가 있으면 디스크에서 로드하고,
# 없으면 Hugging Face 데이터셋에서 500개만 샘플링
_tp = f'{DATA_DIR}/chartqa_train_3000'
train_set = (
    load_from_disk(_tp).select(range(500))
    if os.path.isdir(_tp)
    else load_dataset('HuggingFaceM4/ChartQA', split='train').shuffle(seed=42).select(range(500))
)

# SYSTEM, shrink, to_messages, process_vision_info 는 앞 셀에서 이미 정의됨
# 여기서는 재정의하지 않고 그대로 재사용

def collate_fn(batch):
    # batch는 ChartQA 샘플들의 리스트이며,
    # 이를 모델 입력용 텐서 배치로 변환하는 함수
    imgs = [shrink(b["image"]) for b in batch]  # 이미지 크기 축소 및 RGB 정규화

    # 정답을 포함한 메시지와, 정답을 제외한 프롬프트 메시지를 각각 생성
    full_msgs = [to_messages(im, b["query"], str(b["label"][0])) for im, b in zip(imgs, batch)]
    prompt_msgs = [to_messages(im, b["query"], None) for im, b in zip(imgs, batch)]

    # chat template을 적용해 텍스트 문자열로 변환
    # full_texts: 정답까지 포함한 완성 문장
    # prompt_texts: 정답 직전까지의 프롬프트
    full_texts = [
        processor.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
        for m in full_msgs
    ]
    prompt_texts = [
        processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
        for m in prompt_msgs
    ]

    # 이미지 입력을 processor가 받을 수 있는 형태로 평탄화
    image_inputs = []
    for m in full_msgs:
        ii, _ = process_vision_info(m)
        image_inputs.extend(ii)

    # 텍스트 + 이미지 -> 모델 입력 텐서로 변환
    # padding=True로 배치 길이를 맞춤
    model_inputs = processor(
        text=full_texts,
        images=image_inputs,
        padding=True,
        return_tensors="pt"
    )

    # labels 생성:
    # - input_ids를 복사한 뒤
    # - padding token은 -100으로 마스킹
    # - 정답이 시작되기 전 prompt 부분도 -100으로 마스킹
    # => loss는 정답 토큰에만 계산됨
    labels = model_inputs["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100

    for i in range(len(batch)):
        # prompt 길이 계산: 정답 직전까지의 토큰 수
        plen = processor(
            text=[prompt_texts[i]],
            images=[image_inputs[i]],
            return_tensors="pt"
        )["input_ids"].shape[1]
        labels[i, :plen] = -100

    model_inputs["labels"] = labels
    return model_inputs


# gradient checkpointing 사용 시 캐시 비활성화
model.config.use_cache = False

# 4bit 학습에 맞게 모델을 준비
# - layer norm 등의 안정화
# - gradient checkpointing 지원
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# 비전 쪽 파라미터는 학습하지 않도록 freeze
for n, p in model.named_parameters():
    if 'visual' in n:
        p.requires_grad = False

# LoRA 설정
# - r: rank
# - lora_alpha: scaling factor
# - lora_dropout: regularization
# - target_modules: LoRA를 적용할 선형 레이어 이름들
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
)

# 모델에 LoRA 어댑터 적용
model = get_peft_model(model, lora_config)

# 학습 가능한 파라미터 수 확인
model.print_trainable_parameters()

trainable params: 21,823,488 || all params: 8,788,947,184 || trainable%: 0.2483


In [6]:
# ── 8B QLoRA 짧은 학습 시도 ──────────────────────────────────
from transformers import TrainingArguments, Trainer

# TrainingArguments: 학습 하이퍼파라미터 설정
args = TrainingArguments(
    output_dir=f'{VLM_DIR}/qlora_8b',           # 체크포인트·로그 저장 폴더
    per_device_train_batch_size=1,              # 배치 1(메모리 절약)
    gradient_accumulation_steps=8,              # 8스텝 누적 → 유효 배치 8
    max_steps=30,                               # 수업용 짧게(메모리 확인 목적, 실제 학습 X)
    learning_rate=2e-4,                         # LoRA 학습률
    warmup_steps=max(1, int(0.03 * 30)),        # 워밍업: 전체 3%
    lr_scheduler_type='cosine',                 # 코사인 감소 스케줄
    logging_steps=5,                            # 5스텝마다 로그
    save_strategy='no',                         # 체크포인트 저장 X(메모리 아끼기)
    optim='paged_adamw_8bit',                   # 8bit 페이징 옵티마이저(메모리 효율)
    fp16=True,                                  # 혼합 정밀도 학습(fp16)
    gradient_checkpointing=True,                # 그래디언트 체크포인팅(활성값 메모리↓)
    gradient_checkpointing_kwargs={'use_reentrant': False},  # PyTorch 2.0+ 방식
    remove_unused_columns=False,                # 데이터 로더 컬럼 삭제 X(커스텀 collate_fn 사용)
    report_to='none',                           # 외부 로깅(wandb 등) 비활성화
)

# ── use_cache 완전 비활성화 (gradient checkpointing 경고 방지) ──
def disable_use_cache(m):
    """모델·모든 서브모듈의 config.use_cache=False로 설정.
    
    Qwen3-VL은 계층적 config(text_config 등)을 가져서
    최상위 config만 끄면 부족하다. PEFT 래퍼 안쪽까지 재귀적으로 모두 꺼야
    'use_cache=True is incompatible with gradient_checkpointing' 경고를 없앨 수 있다.
    """
    cfg = getattr(m, 'config', None)
    if cfg is not None and hasattr(cfg, 'use_cache'):
        cfg.use_cache = False
    # text_config 같은 서브컨피그도 처리
    if cfg is not None and hasattr(cfg, 'text_config') and hasattr(cfg.text_config, 'use_cache'):
        cfg.text_config.use_cache = False
    # PEFT 래퍼 안쪽 언어모델 config까지 재귀 탐색
    for sub in m.modules():
        sc = getattr(sub, 'config', None)
        if sc is not None and hasattr(sc, 'use_cache'):
            sc.use_cache = False

disable_use_cache(model)   # ← 이를 호출해야 경고가 사라짐

# Trainer 인스턴스 생성
trainer = Trainer(
    model=model,                    # LoRA로 래핑된 8B 모델
    args=args,                      # 위의 TrainingArguments
    train_dataset=train_set,        # ChartQA 500샘플
    data_collator=collate_fn,       # 배치 텍스트/이미지 → 모델 입력 텐서 변환
    processing_class=processor      # tokenizer + image processor
)

# 피크 메모리 통계 초기화
torch.cuda.reset_peak_memory_stats()

try:
    # 학습 시작
    trainer.train()
    # 학습 완료 시 피크 VRAM 기록
    peak_8b = torch.cuda.max_memory_allocated() / 1024**3
    print(f'\n✅ 8B QLoRA 학습 성공! 피크 VRAM: {peak_8b:.2f} GB')
except torch.cuda.OutOfMemoryError:
    # OOM 발생 시
    peak_8b = float('nan')
    print('\n❌ OOM 발생 — 아래 "4. OOM이 났다면" 섹션의 대처를 적용하세요.')

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
5,0.810343
10,0.706709
15,0.707213
20,0.424432
25,0.474636
30,0.738453



✅ 8B QLoRA 학습 성공! 피크 VRAM: 10.09 GB


### 🔍 결과 해석 — 8B QLoRA 학습 (됐다!)
- `✅ 8B QLoRA 학습 성공! 피크 VRAM: 10.09 GB` → **batch=1 · MAX_PIXELS 절반(128) · grad accum 8 · grad checkpointing · 8bit 옵티마이저**를 총동원하니 8B도 **T4 16GB 안(약 10GB)** 에서 30스텝 학습이 돌았습니다.
- 로드(5.96GB) → 학습 피크(10.09GB): 그래디언트·옵티마이저·활성값이 약 **+4GB** 붙은 것입니다. 4B 학습 피크(5.66GB)보다 확실히 빡빡합니다.
- `tokenizer has new PAD/BOS/EOS ...` 는 정상 안내입니다. 다만 **여유가 적어**, 해상도를 조금만 올리거나 batch를 키우면 OOM이 나기 쉽습니다 → 그래서 "한계선".

## 4. OOM이 났다면 — 대처 순서

위 셀에서 OOM이 나면, **런타임 재시작 후** 다음을 위에서부터 적용:

1. `MAX_PIXELS = 64 * 28 * 28` — 이미지 토큰을 더 줄임(가장 효과 큼)
2. LoRA `r=4` — 어댑터·옵티마이저 메모리↓
3. `max_new_tokens`·시퀀스 길이 점검, 다른 큰 변수 `del` 후 `torch.cuda.empty_cache()`
4. 그래도 안 되면 — **이것이 핵심 교훈**: *T4에서 8B 학습은 한계선*입니다. 그래서 Day 2는 **A100**으로 갑니다.

> 8B는 **추론은 4bit로 T4에서 되지만, 안정적 학습은 빠듯**합니다. 환경의 현실적 상한을 몸으로 확인하는 것이 이 도전의 목적입니다.

## 5. 4B vs 8B 트레이드오프 정리

가중치 메모리는 계산으로, 학습 피크는 측정값으로 채웁니다. (4B 값은 4교시에서 측정한 값을 참고로 기입)

In [7]:
# ── 4B vs 8B 비교표 ─────────────────────────────────────────
def w4bit_gb(n): return n * 0.5 / 1024**3   # 4bit 가중치(GB)
def wfp16_gb(n): return n * 2.0 / 1024**3   # fp16 가중치(GB)

P4, P8 = 4_437_800_000, 8_767_100_000

rows = [
    ('파라미터 수',        '4.44 B',                 '8.77 B'),
    ('가중치 fp16',        f'{wfp16_gb(P4):.1f} GB',  f'{wfp16_gb(P8):.1f} GB'),
    ('가중치 4bit',        f'{w4bit_gb(P4):.1f} GB',  f'{w4bit_gb(P8):.1f} GB'),
    ('T4 추론(4bit)',      '여유',                    '가능(빠듯)'),
    ('T4 QLoRA 학습',      '안정적(권장)',            '한계선(도전)'),
    ('상대 속도',          '빠름',                    '느림(약 1.8~2배)'),
    ('차트 이해 품질',      '좋음',                    '대체로 더 좋음'),
]

print(f'{"항목":<18}{"4B":<16}{"8B":<16}')
print('-' * 50)
for name, a, b in rows:
    print(f'{name:<16}{a:<16}{b:<16}')
print('-' * 50)
try:
    print(f'이번 8B QLoRA 학습 피크(측정): {peak_8b:.2f} GB')
except Exception:
    pass
print('※ 품질은 데이터·학습량에 따라 달라질 수 있는 일반 경향입니다.')

항목                4B              8B              
--------------------------------------------------
파라미터 수          4.44 B          8.77 B          
가중치 fp16        8.3 GB          16.3 GB         
가중치 4bit        2.1 GB          4.1 GB          
T4 추론(4bit)     여유              가능(빠듯)          
T4 QLoRA 학습     안정적(권장)         한계선(도전)         
상대 속도           빠름              느림(약 1.8~2배)    
차트 이해 품질        좋음              대체로 더 좋음        
--------------------------------------------------
이번 8B QLoRA 학습 피크(측정): 10.09 GB
※ 품질은 데이터·학습량에 따라 달라질 수 있는 일반 경향입니다.


### 🔍 결과 해석 — 4B vs 8B
- 4bit 가중치는 4B 2.1GB / 8B 4.1GB로 **둘 다 T4 적재 가능**하지만, **학습 안정성**은 4B(여유) vs 8B(한계선)로 갈립니다. 실측 피크도 4B 5.66GB vs 8B 10.09GB로 차이가 큽니다.
- 8B는 품질이 대체로 더 좋지만 **약 1.8~2배 느립니다**. 그래서 *학습은 4B로 안정적으로, 더 큰 모델·배포 최적화는 더 큰 GPU(Day2 A100)* 라는 전략이 자연스럽습니다.
- ※ 품질은 데이터·학습량에 따라 달라지는 일반 경향입니다.

## 6. Day 1 종합 정리

오늘 한 일 — **학습을 가볍게(QLoRA)**:
1. (1교시) 경량화·양자화 기초, 메모리 수학
2. (2교시) 4bit 로드·베이스라인
3. (3교시) ChartQA 전처리·라벨 마스킹
4. (4교시) **QLoRA 학습** (4B)
5. (5교시) Relaxed Accuracy 평가 · **LoRA 어댑터 Hub push** (무거운 병합은 Day2로 미룸)
6. (6교시) 8B 도전·한계 확인

**핵심 깨달음**: 4bit + LoRA + 8bit 옵티마이저 + grad checkpointing을 조합하면 *T4*로도 VLM 파인튜닝이 됩니다. 단 8B 학습은 한계선이라, 더 큰 모델·배포 최적화는 더 큰 GPU가 필요합니다.

---

## 7. Day 2 예고 — 배포를 가볍게(AWQ + vLLM)

내일은 **RunPod A100 80GB**에서:
- **AWQ**(activation-aware 4bit PTQ)로 모델을 *배포용*으로 양자화
- **vLLM**으로 서빙하고 처리량/지연을 측정
- 오늘 push한 **LoRA 어댑터**를 Day2가 **base 4B에 병합**한 뒤 AWQ, **베이스 8B**는 그대로 AWQ로 변환해 비교
  - (병합은 코랩 RAM 한계 때문에 Day1에서 하지 않고, 메모리가 넉넉한 **Day2 A100**에서 수행합니다.)

### ✅ Day 2 이관 체크리스트
- [ ] 5교시에서 **LoRA 어댑터를 HF Hub(비공개)에 push** 했다 (`<your-username>/Qwen3-VL-4B-ChartQA-lora`)
- [ ] 본인 **HF 토큰**(read 이상; push엔 write)을 Day2에서 쓸 수 있게 준비했다
- [ ] **병합은 Day2(A100)** 에서 수행됨을 이해했다 (Day1은 어댑터만 올림)
- [ ] ChartQA 지표(**Relaxed Accuracy**)와 평가 방식을 이해했다

> ✅ **체크포인트**: ① 8B 4bit 로드 VRAM을 측정했다 ② T4 8B 학습의 한계를 직접 봤다 ③ 4B/8B 트레이드오프를 설명할 수 있다 ④ 이관 체크리스트를 완료했다.